### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable
from sklearn.metrics import r2_score 

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [4]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [71]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()


data = load_data(emp_at_file)
# Get the data for the first session
data = get_session_data(data, 1)

# Map synchrony values to each condition in DataFrame
data['Synchrony'] = data['Condition'].apply(lambda x: sync_results_vector[x-1])

data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])

In [ ]:
# Features-only hierarchical logistic regression
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Model synchrony (mechanism)
model_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + (1 + Synchrony | SubjectID)",
    data=data,
    family="bernoulli"
)

# Full model (features + mechanism)
model_full = bmb.Model(
    "Correct ~ 1 + Synchrony + ContrastHeterogeneity * GridCoarseness + (1 + Synchrony + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Model V1 (Synchrony in V1 model as function of features)
model_v1 = bmb.Model(
    "Synchrony ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness | SubjectID)",
    data=data,
    family="gaussian"
)

Are the factors that determine synchrony among coupled oscillators (frequency detuning and coupling strength) predictive of human ability to segregate a rectangular figure from its background in texture stimuli? 

In [7]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneit

In [19]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['less', 'less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.999 |
|            GridCoarseness            |    less   | 0.999 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.998 |
+--------------------------------------+-----------+-------+

Odds ratios:
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.557 |    0.404     |     0.743     |
|            GridCoarseness            | 0.766 |    0.667     |     0.873     |
| ContrastHeterogeneity:GridCoarseness | 1.266 |    1.120     |     1.432     |
+--------------------------------------+----

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.597,0.151,-0.882,-0.278,0.003,0.003,2216.0,2936.0,1.0
GridCoarseness,-0.269,0.068,-0.406,-0.137,0.001,0.001,4144.0,4233.0,1.0
ContrastHeterogeneity:GridCoarseness,0.234,0.063,0.116,0.360,0.001,0.001,3970.0,4096.0,1.0


Does the synchronization behavior of a biophysical model of V1 predict human ability to segregate a rectangular figure from its background in texture stimuli?

In [9]:
idata_sync = model_sync.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 195 seconds.


In [20]:
predictors = ["Synchrony"]
directions = ['greater']

posterior = posterior_table(idata_sync, predictors, directions)
odds_ratios = OR_table(idata_sync, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_sync, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+-----------+-----------+-------+
| Predictor | direction |   P   |
+-----------+-----------+-------+
| Synchrony |  greater  | 0.997 |
+-----------+-----------+-------+

Odds ratios:
+-----------+-------+--------------+---------------+
| Predictor |  Mean | Lower (2.5%) | Upper (97.5%) |
+-----------+-------+--------------+---------------+
| Synchrony | 2.199 |    1.403     |     3.316     |
+-----------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Synchrony,0.764,0.216,0.352,1.212,0.005,0.004,2029.0,2633.0,1.0


Does model synchrony add predictive power beyond the experimentally manipulated stimulus features? 

In [11]:
idata_full = model_full.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 301 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


In [12]:
# Compare models (LOO)
az.compare({
    "stimulus features": idata_features,
    "synchrony": idata_sync,
    "full model": idata_full,
}, method="BB-pseudo-BMA")

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
full model,0,-3581.594553,32.294748,0.000000,6.402615e-01,28.888927,0.000000,False,log
stimulus features,1,-3582.854919,28.344013,1.260367,3.597385e-01,29.099021,3.026524,False,log
synchrony,2,-3632.604204,16.732052,51.009651,3.187229e-10,27.474419,10.912353,False,log


In [21]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness", "Synchrony"]
directions = ['less', 'less', 'greater', 'greater']

posterior = posterior_table(idata_full, predictors, directions)
odds_ratios = OR_table(idata_full, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_full, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.998 |
|            GridCoarseness            |    less   | 0.998 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 1.000 |
|              Synchrony               |  greater  | 0.868 |
+--------------------------------------+-----------+-------+

Odds ratios:
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.601 |    0.457     |     0.773     |
|            GridCoarseness            | 0.790 |    0.691     |     0.899     |
| ContrastHeterogeneity:GridCoarseness | 1.230 |    1.109     |

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.518,0.134,-0.789,-0.264,0.002,0.003,3567.0,4101.0,1.0
GridCoarseness,-0.237,0.067,-0.371,-0.110,0.001,0.001,6199.0,5503.0,1.0
ContrastHeterogeneity:GridCoarseness,0.206,0.052,0.104,0.311,0.001,0.001,7462.0,5317.0,1.0
Synchrony,0.142,0.140,-0.117,0.434,0.002,0.002,5378.0,4954.0,1.0


In [ ]:
model_v1 = bmb.Model(
    "Synchrony ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness | SubjectID)",
    data=data,
    family="gaussian"
)

In [ ]:
idata_v1 = model_v1.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Synchrony_sigma, Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 41 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


In [36]:
# Prepare design matrix
contrast_heterogeneity = data["ContrastHeterogeneity"].to_numpy()
grid_coarseness = data["GridCoarseness"].to_numpy()
interaction = contrast_heterogeneity * grid_coarseness
design_matrix = np.c_[np.ones_like(contrast_heterogeneity), contrast_heterogeneity, grid_coarseness, interaction]   # columns: Intercept, CH, GC, CHxGC

# Response variable
synchrony = data["Synchrony"].to_numpy()

# Pull posterior draws of fixed effects (coefficients) from the statistical model:
intercept = draw_posterior("Intercept", idata_v1)
beta_contrast_heterogeneity  = draw_posterior("ContrastHeterogeneity", idata_v1)
beta_grid_coarseness  = draw_posterior("GridCoarseness", idata_v1)
beta_interaction = draw_posterior("ContrastHeterogeneity:GridCoarseness", idata_v1)

# Stack coefficients into (n_draws, 4)
coefficients = np.column_stack([intercept, beta_contrast_heterogeneity, beta_grid_coarseness, beta_interaction])  # shape (S, 4)

# Posterior-predictive mean for each draw at population level
synchrony_predicted = coefficients.dot(design_matrix.T)

# Compute R^2 for each draw
explained_variance_draws = np.array([r2_score(synchrony, yhat_s) for yhat_s in synchrony_predicted])

# Summarize Bayesian R^2
explained_variance_mean = float(np.mean(explained_variance_draws))
explained_variance_hdi  = az.hdi(explained_variance_draws, hdi_prob=0.95)
print(f"Bayesian R^2 (population-level): mean={explained_variance_mean:.3f}, 95% CrI=[{explained_variance_hdi[0]:.3f}, {explained_variance_hdi[1]:.3f}]")


Bayesian R^2 (population-level): mean=0.798, 95% CrI=[0.797, 0.798]


### Design Analysis

In [ ]:
rng = np.random.default_rng(1709026616) # Seed for reproducibility
num_simulations = 50
num_subjects_list = [4, 6, 8, 10]

num_available_draws = len(az.extract(idata_features, var_names=["Intercept"]).to_dataframe())

results = []
for num_subjects in num_subjects_list:
    for sim in range(num_simulations):
        draw_index = rng.integers(num_available_draws)
        df_simulated, true_betas = simulate_dataset_from_draw(
            idata_features, data, draw_index, n_subjects=num_subjects, rng=rng
        )
        sim_result = analyze_simulated(df_simulated, true_betas)
        sim_result["num_subjects"] = num_subjects
        sim_result["simulation"] = sim
        results.append(sim_result)

results_df = pd.DataFrame(results)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 78 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:Gr

In [135]:
summary = summarize_design_analysis(results_df)
print(summary)

              Detected CH  Detected GC  Detected Interaction  \
num_subjects                                                   
8                     0.9          0.7                   0.9   

              contrast_heterogeneity_typeS  Type S error GC  Type S error INT  \
num_subjects                                                                    
8                                      0.0              0.0               0.0   

              Type M error CH  Type M error GC  Type M error Interaction  \
num_subjects                                                               
8                        1.07             1.03                      0.98   

              Probability (one-sided) CH  Probability (one-sided) GC  \
num_subjects                                                           
8                                   0.92                        0.95   

              Probability (one-sided) Interaction  
num_subjects                                       
8            